# Phase0: 合成データ生成器 — ナンバー風の画像を「作る」

> 🔰 **やさしい導入**（この回のひとこと）
>
> - **ひとことで言うと**: ナンバープレート風の画像を、プログラムで「作る」回です。難しさ（ぼかし・傾き）も段階的に変えます。
> - **たとえ話**: ナンバー風のスタンプを自動で押し、写真のように汚す工場。正解（位置・文字）が自動で付く。
> - **先に読むと分かりやすい**: [00_basics.ipynb](00_basics.ipynb) の「2-1（画像は数字の並び）」。用語が分からなくなったら、同じNotebookの「用語集」へ。
> - **難しければ**: 各コードの後ろにある「**出力の見方**」の文章だけ読めば、結論は追えます。

> ⏱ 所要時間: 説明を読む 10分 ＋ 実行 1分

## このプロジェクトのゴール（おさらい）
駐車場の**ナンバー読み取り精算システム**の「ミニ版」を作り、**AIはシステムの一部品**であることを自分の手で確かめます。
システムの流れは次のとおりです（AIが担うのは①②だけ）。

```
① ナンバー板を見つける（検出）→ ② 文字を読む（OCR）→ ③ ルールで補正 → ④ 自信が低ければ「要確認」
   → ⑤ 入庫を記録・出庫で照合・料金計算 → ⑥ どれくらい正確かを検証
```

## このNotebookのゴール
**学習に使う画像を、プログラムで自動生成できるようにする。** この回では、AIはまだ登場しません。
次の3つを確認します。
1. 画像と一緒に**正解（板の位置・文字）が自動で付く**こと
2. **難易度**（Easy / Medium / Hard）を変えると、画像がどう変わるか
3. 同じ**シード**なら、必ず**同じ画像**ができること（再現性）

## なぜ「合成画像」なのか
| | 内容 |
|---|---|
| 👍 プライバシー | 実在のナンバーや、実際の車の写真は、**個人につながる情報**。使わずに済む |
| 👍 正解が自動で付く | 自分で描くので、「どこに」「何と書いてあるか」を最初から知っている。人手の注釈作業が不要 |
| 👍 難易度を調整できる | 暗さ・ぼけ・傾き・汚れを、好きな強さで再現できる。**どこで壊れるか**の実験ができる |
| 👍 無限に作れる | 学習データを好きなだけ増やせる（前回の「データ拡張」とは別の増やし方） |
| ⚠ 現実との差 | 書体・背景・質感は実物と違う。**ここでの精度が、実際の性能を保証するわけではない** |

## キーワード
| 用語 | 意味 |
|---|---|
| **合成データ** | プログラムで作った学習用データ |
| **正解ラベル** | 画像に付ける正解の情報（板の位置、文字）。AIは「画像→正解」の対応を学ぶ |
| **bbox（外接矩形）** | 板をぴったり囲む四角形 `[x, y, 幅, 高さ]` |
| **透視変換** | 斜めから見たように、板の形をゆがめる処理 |
| **シード** | 乱数の出方を決める番号。同じ番号なら同じ結果 |
| **難易度** | 画像の劣化の強さの段階（Easy / Medium / Hard） |

> 🔒 ナンバーの番号は**すべてランダム**です。実在の車とたまたま一致することはあっても、狙って作ってはいません。

In [ ]:
import sys
from pathlib import Path

# 正しい環境（プロジェクトの .venv）で動いているか確認する
try:
    import PIL, numpy  # noqa: F401
except ModuleNotFoundError:
    raise SystemExit(
        f"必要なライブラリが見つかりません。今のPython: {sys.executable}\n"
        "→ カーネルを「Python (plate_reader_lab)」（または プロジェクトの .venv/bin/python）に切り替えてください。"
    )

# src/ をimportできるようにする（notebooks/ からでもプロジェクト直下からでも動く）
sys.path.insert(0, str(next(p for p in [Path.cwd() / "src", Path.cwd().parent / "src"] if p.exists())))

import json
import numpy as np
from plate_synth import (DIFFICULTIES, CLEAN, generate_sample, load_labels, random_label, render_plate)
from plate_synth.difficulty import RANGES
from plate_synth.fonts import find_font_path
from plate_synth.viz import draw_label, show_grid

%matplotlib inline
ROOT = next(p for p in [Path.cwd(), Path.cwd().parent] if (p / "src").exists())
print("日本語フォント:", Path(find_font_path()).name)

上のセルは、道具（生成プログラム）の読み込みです。「日本語フォント: ○○」と表示されればOKです。
エラーが出たら、右上のカーネルを **Python (plate_reader_lab)** に切り替えてください。

## 1. プレート1枚を作る

まず、板そのもの（背景なし）を作ります。**自家用・事業用・軽自家用・軽事業用**の4種類を、同じ番号で並べます。
色の違い（地の色と文字の色）に対して、AIが頑健かを後で調べるために、4種類を用意しています。

### コードの読み方
- `random_label(rng)` … ランダムな「内容」（地名・分類番号・かな・一連番号）を作る。
- `render_plate(label)` … 内容から、板の画像を描く。
- `rng = np.random.default_rng(3)` の `3` が**シード**。変えると別の番号になる。

In [ ]:
rng = np.random.default_rng(3)           # シード3: 毎回同じ番号になる
label = random_label(rng)
print("内容:", label["text"])

plates, titles = [], []
for ptype in ["private", "business", "kei_private", "kei_business"]:
    plates.append(render_plate({**label, "plate_type": ptype}))     # 種別だけ変えて描く
    titles.append(ptype)
show_grid(plates, titles, cols=4, size=3.4);

**見るポイント**
- 上段に「地名」と「分類番号（3文字）」、下段に「ひらがな」と「一連番号（4桁）」があります。
- 一連番号は、4桁なら中央にハイフン（例: `12-34`）、3桁以下なら上位が「・」で埋まります（例: `・・12`）。
- ⚠ 書体・色・配置は**学習用の簡略化**です。実際のナンバープレートとは異なります。

## 2. 正解ラベルを見る

画像と一緒に、**正解ラベル**が自動で作られます。1枚ぶんを生成して、中身を見てみましょう。

`generate_sample(seed, difficulty)` が、「シーン画像」と「正解ラベル」の両方を返します。

In [ ]:
s = generate_sample(seed=0, difficulty="easy")
print("画像サイズ:", s.image.size)
# 正解ラベルのうち、主な項目だけ表示
show = {k: s.label[k] for k in ["text", "region", "class_no", "kana", "serial", "plate_type", "bbox", "difficulty", "seed"]}
print(json.dumps(show, ensure_ascii=False, indent=1))

**ラベルの項目**

| 項目 | 意味 |
|---|---|
| `text` | 板に書かれた文字全体（これがOCRの正解になる） |
| `region` / `class_no` / `kana` / `serial` | 地名・分類番号・かな・一連番号 |
| `bbox` | 板を囲む四角形 `[x, y, 幅, 高さ]`（これが検出の正解になる） |
| `corners` | 板の四隅の座標（傾いた板を、まっすぐに直すのに使う） |
| `degradations` | この画像に適用した劣化とその強さ（ぼけ・ノイズなど） |
| `seed` | 生成に使ったシード（同じ画像を再現できる） |

次のセルは、正解の枠（**赤い四角**）と四隅（**水色の点**）を画像の上に描きます。**ラベルが画像と合っているか**を、目で確かめる大切な確認です。

In [ ]:
show_grid([s.image, draw_label(s.image, s.label)], ["生成された画像", "正解ラベルを重ねた画像"], cols=2, size=5.0);

**見るポイント**: 赤い枠が、板をぴったり囲んでいますか？ 四隅の点は、板の角に合っていますか？
板が傾いていると、枠（外接矩形）は板より少し大きくなります。これは正常です。

## 3. 背景を変える

背景は3種類あります（実写真は使いません）。`overrides`（上書き）で、背景の種類だけを指定します。

| 種類 | 内容 |
|---|---|
| `wall` | 壁のような単色＋細かいノイズ |
| `gradient` | 上から下へ色が変わるグラデーション |
| `car` | 車体に見立てた矩形（ライト・バンパー付き） |

In [ ]:
imgs, tt = [], []
for bg in ["wall", "gradient", "car"]:
    # CLEAN は「劣化を全部切った」基準。背景の種類だけ変えて、同じシードで作る
    smp = generate_sample(seed=11, difficulty="easy", overrides={**CLEAN, "bg_type": bg})
    imgs.append(draw_label(smp.image, smp.label)); tt.append(f"背景: {bg}")
show_grid(imgs, tt, cols=3, size=4.2);

## 4. 劣化を1つずつ見る

難易度は、次のような**劣化**の組み合わせで決まります。まず、**1種類ずつ**どう見えるかを確認します。
同じ板に、**1つだけ**劣化をかけて並べます（`CLEAN` ＋ 1項目の上書き）。

| 劣化 | 現実のどんな状況？ |
|---|---|
| 暗い・低コントラスト | 夜間、曇り |
| 逆光・影 | 画面の一部だけ暗い・明るい |
| 汚れ | 泥はね、汚れ |
| 雨滴 | 雨の日のレンズ |
| 隠れ | ワイパー、フレームの線 |
| ぼけ・動きぶれ | ピンぼけ、走行中 |
| 低解像度 | 遠距離、安いカメラ |
| ノイズ・JPEG | 高感度撮影、強い圧縮 |
| 回転・透視 | 斜めから撮影 |

**ねらい**: あとの評価で「どの劣化に弱いか」を**1つずつ**調べる（アブレーション）ため、劣化を**個別に制御できる**作りにしています。

In [ ]:
tests = [
    ("劣化なし（基準）", {}),
    ("暗い brightness=0.45", dict(brightness=0.45)),
    ("低コントラスト 0.5", dict(contrast=0.5)),
    ("逆光 grad=0.6", dict(grad_strength=0.6)),
    ("汚れ 3か所", dict(stain_count=3)),
    ("雨滴 30個", dict(rain_count=30)),
    ("隠れ 2本", dict(occlusion_lines=2)),
    ("ぼけ sigma=2", dict(blur_sigma=2.0)),
    ("動きぶれ 5px", dict(motion_len=5)),
    ("低解像度 1/3", dict(lowres=3.0)),
    ("ノイズ sigma=18", dict(noise_sigma=18.0)),
    ("JPEG品質 15", dict(jpeg_quality=15)),
    ("回転 12度", dict(rot_deg=12.0)),
    ("透視 0.12", dict(persp=0.12)),
]
imgs, tt = [], []
for name, ov in tests:
    smp = generate_sample(seed=21, difficulty="easy", overrides={**CLEAN, "bg_type": "car", "scale": 0.5, **ov})
    imgs.append(smp.image); tt.append(name)
show_grid(imgs, tt, cols=5, size=3.2);

**見るポイント**（実際の画像を見て、考えてみてください）
- **どの劣化が、文字を読みにくくしていますか？** 人間の目で見て、「これは読めない」と感じるものはありますか？
- 回転・透視では、板の**形**が変わります（検出に影響）。ぼけ・ノイズ・低解像度では、**文字の細部**が失われます（OCRに影響）。
- 「ここまでなら読める」という**境界**は、劣化ごとにどのあたりでしょう？ これが、あとで測る「AIの限界」の予想になります。

## 5. 難易度を比べる（Easy / Medium / Hard）

実際の評価では、劣化をランダムに組み合わせた **3つの難易度**を使います。各難易度の**パラメータの範囲**を表にします。

In [ ]:
import pandas as pd
rows = []
for d in DIFFICULTIES:
    r = RANGES[d]
    rows.append({"難易度": d, "板の大きさ(画面幅比)": r["scale"], "回転(度)": r["rot_deg"], "ぼけσ": r["blur_sigma"],
                 "ノイズσ": r["noise"], "明るさ": r["brightness"], "JPEG品質": r["jpeg"],
                 "汚れ(確率,最小,最大)": r["stain"], "紛らわしい物の数": r["distractors"]})
pd.DataFrame(rows).set_index("難易度")

In [ ]:
# 同じシード(0〜3)で、難易度だけ変えて並べる（上段: easy / 中段: medium / 下段: hard）
imgs, tt = [], []
for d in DIFFICULTIES:
    for i in range(4):
        smp = generate_sample(seed=i, difficulty=d)
        imgs.append(draw_label(smp.image, smp.label)); tt.append(f"{d} 〈{smp.label['text']}〉")
show_grid(imgs, tt, cols=4, size=3.4);

**見るポイント**
- **Easy**: 大きく、まっすぐ、くっきり。
- **Medium**: 傾きやぼけ、軽い汚れ、**紛らわしい白い板**（文字のような縦棒がある）が現れる。
- **Hard**: 小さく、強く傾き、暗く、ざらつき、汚れ・雨滴・線で一部が隠れる。**一部は人間にも読みにくい**（これは意図したもの。AIの限界を見るため）。

> 💡 同じシード（0〜3）でも、難易度が違うと画像は違います。板の「内容」は同じで、置き方と劣化が変わります。

## 6. 再現性とデータの分割

### 同じシードなら、同じ画像
学習・評価では、**いつ実行しても同じ画像**であることが大切です（結果を比べるため）。確かめましょう。

In [ ]:
a = generate_sample(seed=123, difficulty="hard")
b = generate_sample(seed=123, difficulty="hard")
same_image = np.array_equal(np.asarray(a.image), np.asarray(b.image))
print("画像が完全に同じ:", same_image, "／ ラベルが同じ:", a.label == b.label)
assert same_image and a.label == b.label

### train / val / test の分け方
画像は、シードの範囲で**3つに分け**、範囲が**重ならない**ようにしています（データリークの防止）。

| セット | 用途 | 作り方 |
|---|---|---|
| **train** | 学習 | 学習中に**その場で生成**（毎回違う画像。無限に増やせる） |
| **val** | しきい値・手法の選択 | **固定シード**で作った固定ファイル（難易度ごとに500枚） |
| **test** | 最終評価 | **固定シード**で作った固定ファイル（難易度ごとに1,000枚）。**最後に1回だけ**使う |

固定の評価セットは、次のコマンドで作ります（約1分。`data/eval/` に出力され、gitには入りません）。

```bash
.venv/bin/python src/make_eval_sets.py
```

作ってあれば、中身を確認できます。

In [ ]:
root = ROOT / "data" / "eval"
if not root.exists():
    print("評価セットが未作成です。ターミナルで  .venv/bin/python src/make_eval_sets.py  を実行してください。")
else:
    rows = []
    for split in ("val", "test"):
        for d in DIFFICULTIES:
            labs = load_labels(root / split / d / "labels.jsonl")
            w = np.mean([l["bbox"][2] for l in labs])
            rows.append({"セット": split, "難易度": d, "枚数": len(labs), "板の幅の平均(px)": round(w, 1),
                         "地名の種類": len({l["region"] for l in labs})})
    display(pd.DataFrame(rows))

**見方**: 難易度が上がるほど、板の**幅の平均が小さく**なっています（遠く・小さく写る）。val と test の枚数も仕様どおりです。

## 7. 現実との差（限界）
- 書体・文字間隔・板の質感は、実物と違います。
- 背景は単純で、実際の駐車場の複雑さ（他の車、看板、影）は再現していません。
- だから、**ここでの高い精度は、実際の性能を保証しません**。このプロジェクトは、**システム設計と検証の考え方**を学ぶためのものです。
- この差のことを **sim2real ギャップ**（合成と現実のずれ）と呼びます。

## 🧪 試してみよう

### 演習1: シードを変えて、いろいろな板を見る
**やること**: 「2. 正解ラベルを見る」のセルの **`seed=0`** を **`seed=7`**、**`seed=100`** などに書き換えて、再実行します。
- **見るポイント**: 地名・番号・かな・板の位置が毎回変わりますか？ 同じ数字なら、毎回同じ結果になりますか？
- **分かること**: シードが**「画像の設計図」**になっていること。

### 演習2: 「人間にも読めない」限界を探す（最重要）
**やること**: 「4. 劣化を1つずつ見る」のセルの **`noise_sigma=18.0`** を **`40.0`**、**`60.0`** に書き換えて再実行します。同様に、**`blur_sigma=2.0`** を **`4.0`** にも。
- **見るポイント**: どの値から、**人間が読めなく**なりますか？
- **分かること**: 劣化には**限界**がある。人間にも読めない画像は、AIにも読めません。そこは**「読み取り不可」として人に回す**設計が必要です（Phase4）。

### 演習3: 文字の種類を増やす（上級）
**やること**: `src/plate_synth/spec.py` を開き、`REGIONS` に **`"名古屋"`** を追加して保存し、**カーネルを再起動**して実行し直します（メニューの「カーネルを再起動」）。
- **見るポイント**: 新しい地名の板が出ますか？ （`generate_sample` を何枚か試す）
- **分かること**: 文字の種類を増やすと、**OCRが覚える文字が増える**（学習が難しくなる）。
- ⚠ 終わったら `"名古屋"` を**削除して戻してください**（評価セットとの整合のため）。

### 考えてみよう（コード不要）
画像を**左右反転**すると、板の文字はどうなりますか？ このプロジェクトで、前回学んだ**左右反転の拡張を使ってはいけない**のはなぜでしょう？
- ヒント: 反転した「沖縄」を、AIは「沖縄」と読むべきでしょうか？ 正解ラベルと画像が**食い違い**ませんか？

---

## ✅ まとめ

- 学習用の画像は、**プログラムで自動生成**できる。**正解（位置・文字）が自動で付く**。
- 難易度（Easy / Medium / Hard）は、劣化（暗さ・ぼけ・傾き・汚れなど）の組み合わせ。劣化は**1つずつ制御**できる。
- **シードを固定**すれば、いつでも同じ画像を再現できる。**train / val / test はシードの範囲を分けて**、データリークを防ぐ。
- 合成データには**現実との差**がある。精度の数字は、実性能の保証ではない。
- ラベルが壊れる処理（**左右反転**など）は使えない。

➡ 次は **Phase1: 非AIのベースライン**。AIを使わず、古典的な画像処理だけで「板を見つける・文字を読む」を試し、**AIなしの限界**を知ります。